# Diagnose Event-Human3.6m coordinates

Compare bimvee's original `importAe` output with the refactored loader. **No Brian2, BrianHF, model, or video rendering.** We are checking whether `y` really occupies only 0–102.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from bimvee.importAe import importAe
from escm.event_input import load_events

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
path = root / "InputData/h36m_sample/cam2_S1_Directions/ch0dvs"
print("Recording:", path)
print("Files:", sorted(p.name for p in path.iterdir()))

## 1. Original importer

Print the returned structure before assuming anything about its channel layout. Find an event dictionary by its data keys (`ts`, `x`, `y`, `pol`), regardless of whether bimvee labels it `dvs`.

In [ ]:
legacy = importAe(filePathOrName=str(path))

def find_event_arrays(obj, name="root"):
    if isinstance(obj, dict):
        if all(k in obj for k in ("ts", "x", "y", "pol")):
            yield name, obj
        else:
            for key, value in obj.items():
                if isinstance(value, (dict, list, tuple)):
                    yield from find_event_arrays(value, f"{name}/{key}")
    elif isinstance(obj, (list, tuple)):
        for i, value in enumerate(obj):
            yield from find_event_arrays(value, f"{name}/{i}")

print("Original result type:", type(legacy).__name__)
print("Top-level keys:", list(legacy) if isinstance(legacy, dict) else "not a dictionary")
legacy_streams = list(find_event_arrays(legacy))
print("Found event arrays:", [(name, len(v["ts"])) for name, v in legacy_streams])
if not legacy_streams:
    print("Original importer did not expose a ts/x/y/pol array; inspect the structure above.")

## 2. Refactored importer and coordinate statistics

Print event coordinate ranges for each available stream. This does not change or rescale the data.

In [ ]:
def report(label, event_dict):
    x, y = np.asarray(event_dict["x"]), np.asarray(event_dict["y"])
    print(f"{label}: {len(x):,} events")
    if len(x):
        print(f"  x: {x.min()}..{x.max()}, y: {y.min()}..{y.max()}")
        print("  y percentiles:", np.percentile(y, [0, 1, 50, 99, 100]))

for name, stream in legacy_streams:
    report("importAe " + name, stream)

try:
    modern = load_events(path, channel="left", width=640, height=480, timestamp_clock_ns=1000)
except ValueError as exc:
    print("Loader configuration error:", exc)
    print("Check the available channel in the exception before retrying.")
    modern = None

if modern is not None:
    report("load_events left", modern["events"])
    for name, stream in legacy_streams:
        if len(stream["x"]) == len(modern["events"]["x"]):
            for axis in ("x", "y"):
                a = np.asarray(stream[axis])
                b = np.asarray(modern["events"][axis])
                print(f"{name}: {axis} identical:", np.array_equal(a, b))

## 3. Spatial event map

Render decoded events directly in sensor coordinates, with a fixed 640 × 480 extent. A horizontal strip here indicates a problem upstream of the ESCM/video renderer.

In [ ]:
view = modern["events"] if modern is not None else (legacy_streams[0][1] if legacy_streams else None)
if view is None:
    raise RuntimeError("No decoded event arrays to plot.")
x, y = np.asarray(view["x"]), np.asarray(view["y"])
canvas = np.zeros((480, 640), dtype=np.uint8)
valid = (x >= 0) & (x < 640) & (y >= 0) & (y < 480)
canvas[y[valid], x[valid]] = 1
fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(canvas, cmap="gray", origin="upper", interpolation="nearest", vmin=0, vmax=1)
ax.set(xlabel="x (pixels)", ylabel="y (pixels)", title="All decoded event locations — no rescaling")
plt.show()